# Ingestão de Documentos Escaneados

O workflow recebe o caminho de uma imagem, transcreve o texto com um modelo multimodal, extrai os campos em um esquema e confere o resultado por regras em código. A topologia é uma corrente de três nós, escrita no grafo e não decidida pelo modelo. Os cinco talões em `documents` vêm do conjunto `Francisco-Cruz/InvoicesReceiptsPT`.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langgraph

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import base64
import json
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import Image
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)

DOCUMENTS = sorted(Path("documents").glob("*.jpg"))
DOCUMENTS

### O documento

Fotos de celular, tortas e com sombra, reduzidas para 1400 pixels no lado maior para cortar custo de imagem sem prejudicar a leitura.

In [ ]:
Image(filename=DOCUMENTS[4], width=380)

### Os nós

O estado entra com o caminho da imagem e ganha uma chave por passo. Nenhum passo apaga o que o anterior escreveu, e é isso que permite auditar de onde veio cada campo.

In [ ]:
class DocumentState(TypedDict):
    path: str
    text: str
    record: dict
    issues: list[str]

O OCR vai no próprio modelo multimodal, com a imagem em base64 como bloco de conteúdo. A alternativa descartada foi pedir os campos direto da imagem, em uma chamada só: funciona, mas joga fora a transcrição, e sem ela não dá para saber se um campo errado veio de leitura ruim ou de interpretação ruim.

In [ ]:
def transcribe(state: DocumentState) -> dict:
    """Lê a imagem com o modelo multimodal e devolve o texto corrido."""
    data = base64.b64encode(Path(state["path"]).read_bytes()).decode()
    message = {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcreva todo o texto deste documento, linha por linha. Não corrija e não comente."},
            {"type": "image", "base64": data, "mime_type": "image/jpeg"},
        ],
    }
    return {"text": model.invoke([message]).content.strip()}

In [ ]:
transcription = transcribe({"path": str(DOCUMENTS[4])})

print(transcription["text"][:600])

O esquema é o contrato da saída, e a `description` de cada campo é instrução. A lista de produtos é um esquema aninhado, porque a quantidade de linhas do talão só se conhece na hora.

In [ ]:
class Product(BaseModel):
    name: str = Field(description="Nome do produto como aparece no documento")
    quantity: float = Field(description="Quantidade comprada")
    price: float = Field(description="Preço total da linha, em euros")


class Receipt(BaseModel):
    seller: str = Field(description="Nome do estabelecimento emissor")
    date: str = Field(description="Data do documento no formato AAAA-MM-DD")
    products: list[Product] = Field(description="Uma entrada por linha de produto do documento")
    total: float = Field(description="Valor total pago, em euros")

In [ ]:
def extract(state: DocumentState) -> dict:
    """Extrai os campos da transcrição no formato do esquema."""
    request = f"Extraia os campos do documento abaixo, sem inventar valor.\n\n{state['text']}"
    return {"record": model.with_structured_output(Receipt).invoke(request).model_dump()}

A conferência não chama o modelo: as regras são de domínio, custam zero token e nunca erram. O nó não corrige nada, só anota em `issues`, e o registro com problema segue marcado para a fila de revisão em vez de sumir.

In [ ]:
def validate(state: DocumentState) -> dict:
    """Confere o registro pelas regras do domínio, sem chamar o modelo."""
    record = state["record"]
    issues = []
    try:
        date.fromisoformat(record["date"])
    except (TypeError, ValueError):
        issues.append("data fora do formato AAAA-MM-DD")
    if not record["products"]:
        issues.append("nenhum produto")
    items = sum(product["price"] for product in record["products"])
    if abs(items - record["total"]) > 0.01:
        issues.append(f"produtos somam {items:.2f} e o total é {record['total']:.2f}")
    return {"issues": issues}

### O grafo

Três nós em ordem, ligados por arestas fixas. O grafo trata um documento por invocação.

In [ ]:
builder = StateGraph(DocumentState)

# nós
builder.add_node("transcribe", transcribe)
builder.add_node("extract", extract)
builder.add_node("validate", validate)

# arestas
builder.add_edge(START, "transcribe")
builder.add_edge("transcribe", "extract")
builder.add_edge("extract", "validate")
builder.add_edge("validate", END)

ingest = builder.compile()
Image(ingest.get_graph().draw_mermaid_png())

In [ ]:
state = ingest.invoke({"path": str(DOCUMENTS[4])})

print(state["issues"])
print(json.dumps(state["record"], ensure_ascii=False, indent=2))

### O lote

Cinco invocações independentes, uma por arquivo, em um laço comum de Python. O CSV é plano, então a lista de produtos vira uma linha por item, com os campos do documento repetidos ao lado.

In [ ]:
records = []
for path in DOCUMENTS:
    state = ingest.invoke({"path": str(path)})
    records.append({"file": path.name, "issues": "; ".join(state["issues"]), **state["record"]})

table = pd.json_normalize(records, record_path="products", meta=["file", "seller", "date", "total", "issues"])
table = table[["file", "seller", "date", "total", "issues", "name", "quantity", "price"]]
table

In [ ]:
output = Path("extracted.csv")
table.to_csv(output, index=False)

print(output.resolve(), len(table), "linhas")